# chain-rule-elementwise — worked example 2: Backward of SiLU/swish out = x * sigmoid(x)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `chain-rule-elementwise`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

SiLU (a.k.a. swish) is `out = x * sigmoid(x)`. It is elementwise, so backprop is again a per-position product `grad_in = grad_out * f'(x)`. Unlike `sigmoid` or `tanh`, SiLU's derivative cannot be written purely in terms of `out`; it depends on `x` through `s = sigmoid(x)`: `f'(x) = s + x*s*(1-s) = s*(1 + x*(1-s))`.

## Worked solution

**Step 1 — forward.** `out = x * sigmoid(x)`, elementwise. The Jacobian is diagonal so we only need the scalar derivative `f'(x)` at each position.

**Step 2 — product rule.** `out = x * s` where `s = sigmoid(x)`. By the product rule, `f'(x) = (d/dx x)*s + x*(d/dx s) = s + x * s'`.

**Step 3 — substitute the sigmoid derivative.** `s' = s*(1-s)`. So `f'(x) = s + x*s*(1-s)`. Factor out `s`: `f'(x) = s * (1 + x*(1-s))`. Either form is correct; the factored form does slightly less arithmetic.

**Step 4 — recompute s from x.** The cached `out` is `x*s`, which doesn't let us recover `s` cleanly, so we recompute `s = t.sigmoid(x)`. This is the honest dependency: SiLU's backward needs `x`, not just `out`.

**Step 5 — chain rule and verify.** `grad_in = grad_out * s * (1 + x*(1 - s))`. We confirm against `torch.nn.functional.silu` + autograd; the difference is at float roundoff.

In [ ]:
def silu_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # out = x * sigmoid(x); f'(x) = s + x*s*(1-s) = s*(1 + x*(1-s)).
    s = t.sigmoid(x)
    return grad_out * s * (1 + x * (1 - s))


t.manual_seed(0)
x = t.randn(3, 5, dtype=t.float64)
grad_out = t.randn(3, 5, dtype=t.float64)
out = x * t.sigmoid(x)
grad_in = silu_back(grad_out, out, x)

xg = x.clone().requires_grad_(True)
t.nn.functional.silu(xg).backward(grad_out)
print("max abs diff vs autograd:", (grad_in - xg.grad).abs().max().item())
print("grad_in shape:", tuple(grad_in.shape))